# Preserve the SrRuO3 plume-dynamics study in Dataerai

This notebook preserves the available raw/processed data, all analysis code,
notebooks with existing outputs, graphs, figures, videos, metadata and reported
key results. The original six plume HDF5 recordings will be added later by DID.

Install `requirements-dataerai.txt` in this kernel, then sign in in a terminal:
`dataerai auth login --server https://beta.dataerai.com`.
For a remote machine add `--device`. See [DATAERAI.md](../DATAERAI.md).


In [ ]:
from pathlib import Path
import os
import sys
import json
REPO_ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents)
                 if (p / "dataerai_preservation.py").is_file())
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))
from dataerai_preservation import (
    build_manifest, read_manifest, connect_client, publish, restore,
    register_recording_dids, RAW_NAMES,
)
STUDY = json.loads((REPO_ROOT / "preservation/study.json").read_text())
RESULTS = json.loads((REPO_ROOT / "preservation/key_results.json").read_text())
STUDY


## Inspect the reported key results

The following values are copied from the original saved notebook outputs,
with source cell references. They have not been recalculated. The full
notebooks retain all existing plots, tables and narrative. The abstract reports
better plume stability and film quality with target conditioning, and degradation
with contamination. Inspect the underlying evidence before reusing a conclusion.


In [ ]:
print(RESULTS["film_thickness"]["verbatim_stdout"])
print(RESULTS["film_thickness"]["note"])
print(RESULTS["afm_roughness_and_holes"]["verbatim_stdout"])
RESULTS["analysis_contract"]


## Inventory and package the available study

Set `SRRUO3_DATA_SOURCE` to the directory containing `AFM/`, `XRD_RSM/`,
`TargetMicroscopy/`, `Plumes/` and `Growth_Parameters.xlsx`. The recovery command
in `DATAERAI.md` can recover the Git-tracked inputs removed from upstream.
Optionally set `SRRUO3_RAW_SOURCE` to a directory containing the original six HDF5
files. On a fresh clone of an already-published catalogue, keep using its IDs.


In [ ]:
DATA_SOURCE = Path(os.environ.get("SRRUO3_DATA_SOURCE", str(REPO_ROOT / "data")))
RAW_SOURCE = os.environ.get("SRRUO3_RAW_SOURCE") or None
if DATA_SOURCE.is_dir():
    manifest = build_manifest(REPO_ROOT, DATA_SOURCE, raw_root=RAW_SOURCE)
else:
    manifest = read_manifest(REPO_ROOT)
    if not all(b.get("asset_id") or b.get("kind") == "files" for b in manifest["bundles"].values()):
        raise FileNotFoundError("Set SRRUO3_DATA_SOURCE to the recovered data directory. See DATAERAI.md.")
print(f"Catalogue: {len(manifest['files'])} files in {len(manifest['bundles'])} bundles")
print("Missing raw recordings:", manifest["missing_required_files"])


## Preserve available data and verify a download

The following cell uploads only bundles that have not already been published,
then downloads each pinned version and checks its SHA-256. It records the asset
and content IDs in `preservation/manifest.json`. The collection keeps its current
access permissions. Share it through Dataerai with the intended collaborators.


In [ ]:
with connect_client() as client:
    manifest = publish(REPO_ROOT, client)
manifest["publication"]


## Register the raw recording DIDs when available

Fill the dictionary below after uploading each raw HDF5 file to Dataerai.
Use its exact filename as the key and its DID as the value. Each asset must
contain just that file. Leave the mapping empty until uploads are available.
Registration downloads the actual data and pins its content version and checksum.
Partial registration is resumable; notebooks 4 and 5 require all six recordings.


In [ ]:
RECORDING_DIDS = {
    # "YG063_YichenGuo_08042024.h5": "did:dataerai:...",
    # "YG065_YichenGuo_09102024.h5": "did:dataerai:...",
    # "YG066_YichenGuo_09112024.h5": "did:dataerai:...",
    # "YG067_YichenGuo_09122024.h5": "did:dataerai:...",
    # "YG068_YichenGuo_09132024.h5": "did:dataerai:...",
    # "YG069_YichenGuo_09152024.h5": "did:dataerai:...",
}
if RECORDING_DIDS:
    with connect_client() as client:
        manifest = register_recording_dids(REPO_ROOT, client, RECORDING_DIDS)
        manifest = publish(REPO_ROOT, client)
print("Preservation status:", manifest["completeness"])
print("Still needed:", manifest["missing_required_files"])


## Verify access from Dataerai

Restore the growth-parameter workbook into a separate folder and display its
available sheets. Every analysis notebook uses the same verified restoration
path for its own inputs. Commit the updated catalogue after registering DIDs.


In [ ]:
VERIFY_FOLDER = REPO_ROOT / ".dataerai" / "verification"
with connect_client() as client:
    restore(REPO_ROOT, client, {"study-data"}, VERIFY_FOLDER)
import pandas as pd
workbook = pd.ExcelFile(VERIFY_FOLDER / "data/Growth_Parameters.xlsx")
print(workbook.sheet_names)
for sheet in workbook.sheet_names:
    display(pd.read_excel(workbook, sheet_name=sheet))
